# Problem Statement

Predicting the outcome of a T20 cricket match is a challenging task due to the dynamic nature of the game. The objective of this project is to build a Machine Learning model that can predict the probability of the batting team winning a match during the second innings based on the current match situation.

The model will use real-time match features such as batting team, bowling team, venue, target score, runs left, balls left, wickets left, current run rate, and required run rate to estimate the likelihood of victory. By leveraging historical IPL ball-by-ball data, the project aims to identify the key factors that influence match outcomes and develop an accurate win prediction system.

This project demonstrates the application of data cleaning, feature engineering, classification algorithms, and model evaluation techniques in a real-world sports analytics problem.

In [303]:
import pandas as pd
import numpy as np

matches = pd.read_csv('ipldata.csv')
delivery = pd.read_csv('delivery.csv')

print(matches.shape)
print(delivery.shape)

(636, 18)
(150460, 21)


# Data Audit

In [304]:
# columns in the data set
print(pd.Series(matches.columns.tolist()))
print("\n")
print(pd.Series(delivery.columns.tolist()))

0                  id
1              season
2                city
3                date
4               team1
5               team2
6         toss_winner
7       toss_decision
8              result
9          dl_applied
10             winner
11        win_by_runs
12     win_by_wickets
13    player_of_match
14              venue
15            umpire1
16            umpire2
17            umpire3
dtype: object


0             match_id
1               inning
2         batting_team
3         bowling_team
4                 over
5                 ball
6              batsman
7          non_striker
8               bowler
9        is_super_over
10           wide_runs
11            bye_runs
12         legbye_runs
13         noball_runs
14        penalty_runs
15        batsman_runs
16          extra_runs
17          total_runs
18    player_dismissed
19      dismissal_kind
20             fielder
dtype: object


## Info

In [305]:
matches.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 636 entries, 0 to 635
Data columns (total 18 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   id               636 non-null    int64  
 1   season           636 non-null    int64  
 2   city             629 non-null    object 
 3   date             636 non-null    object 
 4   team1            636 non-null    object 
 5   team2            636 non-null    object 
 6   toss_winner      636 non-null    object 
 7   toss_decision    636 non-null    object 
 8   result           636 non-null    object 
 9   dl_applied       636 non-null    int64  
 10  winner           633 non-null    object 
 11  win_by_runs      636 non-null    int64  
 12  win_by_wickets   636 non-null    int64  
 13  player_of_match  633 non-null    object 
 14  venue            636 non-null    object 
 15  umpire1          635 non-null    object 
 16  umpire2          635 non-null    object 
 17  umpire3         

In [306]:
delivery.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 150460 entries, 0 to 150459
Data columns (total 21 columns):
 #   Column            Non-Null Count   Dtype 
---  ------            --------------   ----- 
 0   match_id          150460 non-null  int64 
 1   inning            150460 non-null  int64 
 2   batting_team      150460 non-null  object
 3   bowling_team      150460 non-null  object
 4   over              150460 non-null  int64 
 5   ball              150460 non-null  int64 
 6   batsman           150460 non-null  object
 7   non_striker       150460 non-null  object
 8   bowler            150460 non-null  object
 9   is_super_over     150460 non-null  int64 
 10  wide_runs         150460 non-null  int64 
 11  bye_runs          150460 non-null  int64 
 12  legbye_runs       150460 non-null  int64 
 13  noball_runs       150460 non-null  int64 
 14  penalty_runs      150460 non-null  int64 
 15  batsman_runs      150460 non-null  int64 
 16  extra_runs        150460 non-null  int

## Finding Null 

In [307]:
matches.isnull().sum()

id                   0
season               0
city                 7
date                 0
team1                0
team2                0
toss_winner          0
toss_decision        0
result               0
dl_applied           0
winner               3
win_by_runs          0
win_by_wickets       0
player_of_match      3
venue                0
umpire1              1
umpire2              1
umpire3            636
dtype: int64

In [308]:
delivery.isnull().sum()

match_id                 0
inning                   0
batting_team             0
bowling_team             0
over                     0
ball                     0
batsman                  0
non_striker              0
bowler                   0
is_super_over            0
wide_runs                0
bye_runs                 0
legbye_runs              0
noball_runs              0
penalty_runs             0
batsman_runs             0
extra_runs               0
total_runs               0
player_dismissed    143022
dismissal_kind      143022
fielder             145091
dtype: int64

## Duplicates

In [309]:
matches.duplicated().sum()

np.int64(0)

In [310]:
delivery.duplicated().sum()

np.int64(1)

## Data Cleaning Observations

### Matches Dataset
- 7 missing values in `city`.
- 3 missing values in `winner`.
- 3 missing values in `player_of_match`.
- `umpire3` contains all missing values and can be dropped.
- No duplicate records found.

### Delivery Dataset
- Missing values in `player_dismissed`, `dismissal_kind`, and `fielder` are expected because wickets do not occur on every ball.
- 1 duplicate record found.

### Cleaning Plan
- Remove duplicate records.
- Drop `umpire3`.
- Investigate matches with missing winners.
- Check DL-affected matches and Super Overs before model training.

In [311]:
matches['dl_applied'].value_counts()

dl_applied
0    620
1     16
Name: count, dtype: int64

In [312]:
delivery['is_super_over'].value_counts()

is_super_over
0    150379
1        81
Name: count, dtype: int64

## Cricket-Specific Data Investigation

### DL-Affected Matches
- 16 matches were affected by the Duckworth-Lewis method.
- These matches represent only a small portion of the dataset.
- Since targets are modified due to weather interruptions, these matches will be excluded from model training.

### Super Overs
- 81 deliveries belong to Super Overs.
- Super Overs represent exceptional match situations and do not reflect normal T20 chase conditions.
- These deliveries will be excluded from the analysis.

# DATA CLEANING

In [313]:
matches = matches[matches['dl_applied'] == 0]  # Remove DL affected matches
delivery = delivery[delivery['is_super_over'] == 0]  # Remove Super Over deliveries

delivery = delivery.drop_duplicates()   # Remove duplicate delivery row
matches = matches.drop(columns=['umpire3']) # Drop the umpire3 column

In [314]:
print(matches.shape)
print(delivery.shape)

(620, 17)
(150378, 21)


In [395]:
# Rising Pune Supergiant and Rising Pune Supergiants both are same team
final_data.replace(
    'Rising Pune Supergiants',
    'Rising Pune Supergiant',
    inplace=True
)
sorted(final_data['batting_team'].unique())

['Chennai Super Kings',
 'Deccan Chargers',
 'Delhi Capitals',
 'Gujarat Lions',
 'Kochi Tuskers Kerala',
 'Kolkata Knight Riders',
 'Mumbai Indians',
 'Pune Warriors',
 'Punjab Kings',
 'Rajasthan Royals',
 'Rising Pune Supergiant',
 'Royal Challengers Bangalore',
 'Sunrisers Hyderabad']

In [396]:
# Standarize the team name
final_data['batting_team'] = final_data['batting_team'].replace({
    'Delhi Daredevils': 'Delhi Capitals',
    'Kings XI Punjab': 'Punjab Kings'
})

final_data['bowling_team'] = final_data['bowling_team'].replace({
    'Delhi Daredevils': 'Delhi Capitals',
    'Kings XI Punjab': 'Punjab Kings'
})

In [397]:
# team list
sorted(final_data['batting_team'].unique())

['Chennai Super Kings',
 'Deccan Chargers',
 'Delhi Capitals',
 'Gujarat Lions',
 'Kochi Tuskers Kerala',
 'Kolkata Knight Riders',
 'Mumbai Indians',
 'Pune Warriors',
 'Punjab Kings',
 'Rajasthan Royals',
 'Rising Pune Supergiant',
 'Royal Challengers Bangalore',
 'Sunrisers Hyderabad']

# FEATURES MAKING

### TARGET CONSTRUCTION

In [315]:
first_inning = delivery[delivery['inning'] == 1]

target_df = first_inning.groupby('match_id')['total_runs'].sum().reset_index()
target_df['target'] = target_df['total_runs'] + 1

target_df.head()

,match_id,total_runs,target
0,1,207,208
1,2,184,185
2,3,183,184
3,4,163,164
4,5,157,158


In [316]:
# Merging the target_df with the matches dataset

matches_df = matches.merge(target_df[['match_id', 'target']],left_on='id',right_on='match_id')

In [317]:
matches_df.shape

(620, 19)

In [318]:
# Useful Column

matches_df[['id','venue','team1','team2','target','winner']].head()

,id,venue,team1,team2,target,winner
0,1,"Rajiv Gandhi International Stadium, Uppal",Sunrisers Hyderabad,Royal Challengers Bangalore,208,Sunrisers Hyderabad
1,2,Maharashtra Cricket Association Stadium,Mumbai Indians,Rising Pune Supergiant,185,Rising Pune Supergiant
2,3,Saurashtra Cricket Association Stadium,Gujarat Lions,Kolkata Knight Riders,184,Kolkata Knight Riders
3,4,Holkar Cricket Stadium,Rising Pune Supergiant,Kings XI Punjab,164,Kings XI Punjab
4,5,M Chinnaswamy Stadium,Royal Challengers Bangalore,Delhi Daredevils,158,Royal Challengers Bangalore


### Target Score Construction

To predict the outcome of a chase, the target score for each match was calculated using the first innings total.

Target Score = First Innings Runs + 1

The calculated target was then merged with the match-level dataset to provide the required context for second innings win prediction.

In [319]:
# Information about teh second inning

second_inning = delivery[delivery['inning'] == 2].copy()
print(second_inning.shape)
second_inning.head()

(72350, 21)


,match_id,inning,batting_team,bowling_team,over,ball,batsman,non_striker,bowler,is_super_over,...,bye_runs,legbye_runs,noball_runs,penalty_runs,batsman_runs,extra_runs,total_runs,player_dismissed,dismissal_kind,fielder
125,1,2,Royal Challengers Bangalore,Sunrisers Hyderabad,1,1,CH Gayle,Mandeep Singh,A Nehra,0,...,0,0,0,0,1,0,1,NaN,NaN,NaN
126,1,2,Royal Challengers Bangalore,Sunrisers Hyderabad,1,2,Mandeep Singh,CH Gayle,A Nehra,0,...,0,0,0,0,0,0,0,NaN,NaN,NaN
127,1,2,Royal Challengers Bangalore,Sunrisers Hyderabad,1,3,Mandeep Singh,CH Gayle,A Nehra,0,...,0,0,0,0,0,0,0,NaN,NaN,NaN
128,1,2,Royal Challengers Bangalore,Sunrisers Hyderabad,1,4,Mandeep Singh,CH Gayle,A Nehra,0,...,0,0,0,0,2,0,2,NaN,NaN,NaN
129,1,2,Royal Challengers Bangalore,Sunrisers Hyderabad,1,5,Mandeep Singh,CH Gayle,A Nehra,0,...,0,0,0,0,4,0,4,NaN,NaN,NaN


In [320]:
# Merging the second_inning with matches_def data set
second_innings = second_inning.merge(
    matches_df[['match_id','venue','winner','target']],on='match_id')

second_innings.head()

,match_id,inning,batting_team,bowling_team,over,ball,batsman,non_striker,bowler,is_super_over,...,penalty_runs,batsman_runs,extra_runs,total_runs,player_dismissed,dismissal_kind,fielder,venue,winner,target
0,1,2,Royal Challengers Bangalore,Sunrisers Hyderabad,1,1,CH Gayle,Mandeep Singh,A Nehra,0,...,0,1,0,1,NaN,NaN,NaN,"Rajiv Gandhi International Stadium, Uppal",Sunrisers Hyderabad,208
1,1,2,Royal Challengers Bangalore,Sunrisers Hyderabad,1,2,Mandeep Singh,CH Gayle,A Nehra,0,...,0,0,0,0,NaN,NaN,NaN,"Rajiv Gandhi International Stadium, Uppal",Sunrisers Hyderabad,208
2,1,2,Royal Challengers Bangalore,Sunrisers Hyderabad,1,3,Mandeep Singh,CH Gayle,A Nehra,0,...,0,0,0,0,NaN,NaN,NaN,"Rajiv Gandhi International Stadium, Uppal",Sunrisers Hyderabad,208
3,1,2,Royal Challengers Bangalore,Sunrisers Hyderabad,1,4,Mandeep Singh,CH Gayle,A Nehra,0,...,0,2,0,2,NaN,NaN,NaN,"Rajiv Gandhi International Stadium, Uppal",Sunrisers Hyderabad,208
4,1,2,Royal Challengers Bangalore,Sunrisers Hyderabad,1,5,Mandeep Singh,CH Gayle,A Nehra,0,...,0,4,0,4,NaN,NaN,NaN,"Rajiv Gandhi International Stadium, Uppal",Sunrisers Hyderabad,208


In [321]:
second_innings.columns.tolist()

['match_id',
 'inning',
 'batting_team',
 'bowling_team',
 'over',
 'ball',
 'batsman',
 'non_striker',
 'bowler',
 'is_super_over',
 'wide_runs',
 'bye_runs',
 'legbye_runs',
 'noball_runs',
 'penalty_runs',
 'batsman_runs',
 'extra_runs',
 'total_runs',
 'player_dismissed',
 'dismissal_kind',
 'fielder',
 'venue',
 'winner',
 'target']

## Second Innings Selection

Since the objective of this project is to predict the outcome of a run chase, only second innings deliveries were selected.

The second innings data was merged with match-level information such as target score, venue, and match winner to provide the required context for prediction.

## CURRENT SCORE

In [322]:
second_innings['current_score'] = (second_innings.groupby('match_id')['total_runs'].cumsum())

second_innings[['match_id','over','ball','total_runs','current_score']].head(10)

,match_id,over,ball,total_runs,current_score
0,1,1,1,1,1
1,1,1,2,0,1
2,1,1,3,0,1
3,1,1,4,2,3
4,1,1,5,4,7
5,1,1,6,4,11
6,1,2,1,0,11
7,1,2,2,0,11
8,1,2,3,1,12
9,1,2,4,0,12


## Run Left

In [323]:
second_innings['runs_left'] = (second_innings['target'] - second_innings['current_score'])
second_innings.head()

,match_id,inning,batting_team,bowling_team,over,ball,batsman,non_striker,bowler,is_super_over,...,extra_runs,total_runs,player_dismissed,dismissal_kind,fielder,venue,winner,target,current_score,runs_left
0,1,2,Royal Challengers Bangalore,Sunrisers Hyderabad,1,1,CH Gayle,Mandeep Singh,A Nehra,0,...,0,1,NaN,NaN,NaN,"Rajiv Gandhi International Stadium, Uppal",Sunrisers Hyderabad,208,1,207
1,1,2,Royal Challengers Bangalore,Sunrisers Hyderabad,1,2,Mandeep Singh,CH Gayle,A Nehra,0,...,0,0,NaN,NaN,NaN,"Rajiv Gandhi International Stadium, Uppal",Sunrisers Hyderabad,208,1,207
2,1,2,Royal Challengers Bangalore,Sunrisers Hyderabad,1,3,Mandeep Singh,CH Gayle,A Nehra,0,...,0,0,NaN,NaN,NaN,"Rajiv Gandhi International Stadium, Uppal",Sunrisers Hyderabad,208,1,207
3,1,2,Royal Challengers Bangalore,Sunrisers Hyderabad,1,4,Mandeep Singh,CH Gayle,A Nehra,0,...,0,2,NaN,NaN,NaN,"Rajiv Gandhi International Stadium, Uppal",Sunrisers Hyderabad,208,3,205
4,1,2,Royal Challengers Bangalore,Sunrisers Hyderabad,1,5,Mandeep Singh,CH Gayle,A Nehra,0,...,0,4,NaN,NaN,NaN,"Rajiv Gandhi International Stadium, Uppal",Sunrisers Hyderabad,208,7,201


## Balls Bowled

In [324]:
# number of balls bowled
second_innings['balls_bowled'] = ((second_innings['over']-1)*6 + second_innings['ball'])
second_innings.head(10)

,match_id,inning,batting_team,bowling_team,over,ball,batsman,non_striker,bowler,is_super_over,...,total_runs,player_dismissed,dismissal_kind,fielder,venue,winner,target,current_score,runs_left,balls_bowled
0,1,2,Royal Challengers Bangalore,Sunrisers Hyderabad,1,1,CH Gayle,Mandeep Singh,A Nehra,0,...,1,NaN,NaN,NaN,"Rajiv Gandhi International Stadium, Uppal",Sunrisers Hyderabad,208,1,207,1
1,1,2,Royal Challengers Bangalore,Sunrisers Hyderabad,1,2,Mandeep Singh,CH Gayle,A Nehra,0,...,0,NaN,NaN,NaN,"Rajiv Gandhi International Stadium, Uppal",Sunrisers Hyderabad,208,1,207,2
2,1,2,Royal Challengers Bangalore,Sunrisers Hyderabad,1,3,Mandeep Singh,CH Gayle,A Nehra,0,...,0,NaN,NaN,NaN,"Rajiv Gandhi International Stadium, Uppal",Sunrisers Hyderabad,208,1,207,3
3,1,2,Royal Challengers Bangalore,Sunrisers Hyderabad,1,4,Mandeep Singh,CH Gayle,A Nehra,0,...,2,NaN,NaN,NaN,"Rajiv Gandhi International Stadium, Uppal",Sunrisers Hyderabad,208,3,205,4
4,1,2,Royal Challengers Bangalore,Sunrisers Hyderabad,1,5,Mandeep Singh,CH Gayle,A Nehra,0,...,4,NaN,NaN,NaN,"Rajiv Gandhi International Stadium, Uppal",Sunrisers Hyderabad,208,7,201,5
5,1,2,Royal Challengers Bangalore,Sunrisers Hyderabad,1,6,Mandeep Singh,CH Gayle,A Nehra,0,...,4,NaN,NaN,NaN,"Rajiv Gandhi International Stadium, Uppal",Sunrisers Hyderabad,208,11,197,6
6,1,2,Royal Challengers Bangalore,Sunrisers Hyderabad,2,1,CH Gayle,Mandeep Singh,B Kumar,0,...,0,NaN,NaN,NaN,"Rajiv Gandhi International Stadium, Uppal",Sunrisers Hyderabad,208,11,197,7
7,1,2,Royal Challengers Bangalore,Sunrisers Hyderabad,2,2,CH Gayle,Mandeep Singh,B Kumar,0,...,0,NaN,NaN,NaN,"Rajiv Gandhi International Stadium, Uppal",Sunrisers Hyderabad,208,11,197,8
8,1,2,Royal Challengers Bangalore,Sunrisers Hyderabad,2,3,CH Gayle,Mandeep Singh,B Kumar,0,...,1,NaN,NaN,NaN,"Rajiv Gandhi International Stadium, Uppal",Sunrisers Hyderabad,208,12,196,9
9,1,2,Royal Challengers Bangalore,Sunrisers Hyderabad,2,4,Mandeep Singh,CH Gayle,B Kumar,0,...,0,NaN,NaN,NaN,"Rajiv Gandhi International Stadium, Uppal",Sunrisers Hyderabad,208,12,196,10


## Balls Left

In [325]:
second_innings['balls_left'] = (120 - second_innings['balls_bowled'])
second_innings.head()

,match_id,inning,batting_team,bowling_team,over,ball,batsman,non_striker,bowler,is_super_over,...,player_dismissed,dismissal_kind,fielder,venue,winner,target,current_score,runs_left,balls_bowled,balls_left
0,1,2,Royal Challengers Bangalore,Sunrisers Hyderabad,1,1,CH Gayle,Mandeep Singh,A Nehra,0,...,NaN,NaN,NaN,"Rajiv Gandhi International Stadium, Uppal",Sunrisers Hyderabad,208,1,207,1,119
1,1,2,Royal Challengers Bangalore,Sunrisers Hyderabad,1,2,Mandeep Singh,CH Gayle,A Nehra,0,...,NaN,NaN,NaN,"Rajiv Gandhi International Stadium, Uppal",Sunrisers Hyderabad,208,1,207,2,118
2,1,2,Royal Challengers Bangalore,Sunrisers Hyderabad,1,3,Mandeep Singh,CH Gayle,A Nehra,0,...,NaN,NaN,NaN,"Rajiv Gandhi International Stadium, Uppal",Sunrisers Hyderabad,208,1,207,3,117
3,1,2,Royal Challengers Bangalore,Sunrisers Hyderabad,1,4,Mandeep Singh,CH Gayle,A Nehra,0,...,NaN,NaN,NaN,"Rajiv Gandhi International Stadium, Uppal",Sunrisers Hyderabad,208,3,205,4,116
4,1,2,Royal Challengers Bangalore,Sunrisers Hyderabad,1,5,Mandeep Singh,CH Gayle,A Nehra,0,...,NaN,NaN,NaN,"Rajiv Gandhi International Stadium, Uppal",Sunrisers Hyderabad,208,7,201,5,115


## Wickets Left

In [326]:
second_innings['wickets'] = (second_innings['player_dismissed'].notna().astype(int))


second_innings['wickets_fallen'] = second_innings.groupby('match_id')['wickets'].cumsum()

second_innings['wickets_left'] = (10 - second_innings['wickets_fallen'])

second_innings[['match_id','over','ball','player_dismissed','wickets_fallen','wickets_left']].head(10)
   

,match_id,over,ball,player_dismissed,wickets_fallen,wickets_left
0,1,1,1,NaN,0,10
1,1,1,2,NaN,0,10
2,1,1,3,NaN,0,10
3,1,1,4,NaN,0,10
4,1,1,5,NaN,0,10
5,1,1,6,NaN,0,10
6,1,2,1,NaN,0,10
7,1,2,2,NaN,0,10
8,1,2,3,NaN,0,10
9,1,2,4,NaN,0,10


## Current Run Rate

In [327]:
# Current run rate meaning 

second_innings['current_rr'] = (second_innings['current_score']*6)/ second_innings['balls_bowled']
second_innings[['match_id','over','ball','player_dismissed','wickets_fallen','wickets_left','current_rr']].head(10)

,match_id,over,ball,player_dismissed,wickets_fallen,wickets_left,current_rr
0,1,1,1,NaN,0,10,6.000000
1,1,1,2,NaN,0,10,3.000000
2,1,1,3,NaN,0,10,2.000000
3,1,1,4,NaN,0,10,4.500000
4,1,1,5,NaN,0,10,8.400000
5,1,1,6,NaN,0,10,11.000000
6,1,2,1,NaN,0,10,9.428571
7,1,2,2,NaN,0,10,8.250000
8,1,2,3,NaN,0,10,8.000000
9,1,2,4,NaN,0,10,7.200000


## Required Run rate

In [328]:
# run_left * 6/balls_left

second_innings['required_rr'] = (second_innings['runs_left'] *6)/(second_innings['balls_left'])
second_innings[['match_id','over','ball','player_dismissed','wickets_fallen','wickets_left','current_rr','required_rr']].head(10)

,match_id,over,ball,player_dismissed,wickets_fallen,wickets_left,current_rr,required_rr
0,1,1,1,NaN,0,10,6.000000,10.436975
1,1,1,2,NaN,0,10,3.000000,10.525424
2,1,1,3,NaN,0,10,2.000000,10.615385
3,1,1,4,NaN,0,10,4.500000,10.603448
4,1,1,5,NaN,0,10,8.400000,10.486957
5,1,1,6,NaN,0,10,11.000000,10.368421
6,1,2,1,NaN,0,10,9.428571,10.460177
7,1,2,2,NaN,0,10,8.250000,10.553571
8,1,2,3,NaN,0,10,8.000000,10.594595
9,1,2,4,NaN,0,10,7.200000,10.690909


## Run Rate Features

Two important run-rate based features were created:

- Current Run Rate (CRR): Measures the scoring rate of the chasing team.
- Required Run Rate (RRR): Measures the scoring rate needed to achieve the target.

These features capture the pressure and momentum of a run chase and are expected to be highly influential in predicting match outcomes.

In [329]:
# set the target variable 

second_innings['result'] = np.where(
    second_innings['batting_team'] == second_innings['winner'],1,0
)



In [330]:
second_innings[['batting_team','winner','result']].head(10)

,batting_team,winner,result
0,Royal Challengers Bangalore,Sunrisers Hyderabad,0
1,Royal Challengers Bangalore,Sunrisers Hyderabad,0
2,Royal Challengers Bangalore,Sunrisers Hyderabad,0
3,Royal Challengers Bangalore,Sunrisers Hyderabad,0
4,Royal Challengers Bangalore,Sunrisers Hyderabad,0
5,Royal Challengers Bangalore,Sunrisers Hyderabad,0
6,Royal Challengers Bangalore,Sunrisers Hyderabad,0
7,Royal Challengers Bangalore,Sunrisers Hyderabad,0
8,Royal Challengers Bangalore,Sunrisers Hyderabad,0
9,Royal Challengers Bangalore,Sunrisers Hyderabad,0


In [331]:
second_innings['result'].value_counts()

result
1    37483
0    33887
Name: count, dtype: int64

In [332]:
# check the result in different random rows
second_innings[['batting_team','winner','result']].sample(20)

,batting_team,winner,result
51639,Chennai Super Kings,Mumbai Indians,0
58172,Chennai Super Kings,Chennai Super Kings,1
45775,Royal Challengers Bangalore,Royal Challengers Bangalore,1
13458,Chennai Super Kings,Delhi Daredevils,0
47850,Delhi Daredevils,Delhi Daredevils,1
2931,Gujarat Lions,Kings XI Punjab,0
10924,Deccan Chargers,Delhi Daredevils,0
60349,Rajasthan Royals,Rajasthan Royals,1
50999,Delhi Daredevils,Pune Warriors,0
47281,Pune Warriors,Delhi Daredevils,0


In [333]:
# choosing the required columns

second_innings.columns.tolist()

['match_id',
 'inning',
 'batting_team',
 'bowling_team',
 'over',
 'ball',
 'batsman',
 'non_striker',
 'bowler',
 'is_super_over',
 'wide_runs',
 'bye_runs',
 'legbye_runs',
 'noball_runs',
 'penalty_runs',
 'batsman_runs',
 'extra_runs',
 'total_runs',
 'player_dismissed',
 'dismissal_kind',
 'fielder',
 'venue',
 'winner',
 'target',
 'current_score',
 'runs_left',
 'balls_bowled',
 'balls_left',
 'wickets',
 'wickets_fallen',
 'wickets_left',
 'current_rr',
 'required_rr',
 'result']

## Target Variable Creation

The target variable was created based on the final match outcome.

- 1 indicates that the chasing team won the match.
- 0 indicates that the chasing team lost the match.

The resulting dataset contains a balanced distribution of winning and losing examples, making it suitable for classification modeling.

# Final Dataset Preparation

In [334]:
final_data = second_innings[
    [
        'match_id',
         'venue',
        'batting_team',
        'bowling_team',
        'runs_left',
        'balls_left',
        'wickets_left',
        'current_rr',
        'required_rr',
        'result'
    ]
]

In [335]:
final_data.shape

(71370, 10)

In [336]:
final_data.head()

,match_id,venue,batting_team,bowling_team,runs_left,balls_left,wickets_left,current_rr,required_rr,result
0,1,"Rajiv Gandhi International Stadium, Uppal",Royal Challengers Bangalore,Sunrisers Hyderabad,207,119,10,6.0,10.436975,0
1,1,"Rajiv Gandhi International Stadium, Uppal",Royal Challengers Bangalore,Sunrisers Hyderabad,207,118,10,3.0,10.525424,0
2,1,"Rajiv Gandhi International Stadium, Uppal",Royal Challengers Bangalore,Sunrisers Hyderabad,207,117,10,2.0,10.615385,0
3,1,"Rajiv Gandhi International Stadium, Uppal",Royal Challengers Bangalore,Sunrisers Hyderabad,205,116,10,4.5,10.603448,0
4,1,"Rajiv Gandhi International Stadium, Uppal",Royal Challengers Bangalore,Sunrisers Hyderabad,201,115,10,8.4,10.486957,0


In [337]:
# missing values checvk
final_data.isnull().sum()

match_id        0
venue           0
batting_team    0
bowling_team    0
runs_left       0
balls_left      0
wickets_left    0
current_rr      0
required_rr     5
result          0
dtype: int64

In [338]:
final_data[final_data['required_rr'].isnull()]

,match_id,venue,batting_team,bowling_team,runs_left,balls_left,wickets_left,current_rr,required_rr,result
15578,142,St George's Park,Kings XI Punjab,Kolkata Knight Riders,0,0,6,7.70,NaN,1
18291,166,SuperSport Park,Kolkata Knight Riders,Chennai Super Kings,0,0,7,9.45,NaN,1
37238,334,"MA Chidambaram Stadium, Chepauk",Chennai Super Kings,Rajasthan Royals,0,0,7,7.35,NaN,1
39773,355,Wankhede Stadium,Mumbai Indians,Chennai Super Kings,0,0,2,8.70,NaN,1
44572,396,"MA Chidambaram Stadium, Chepauk",Chennai Super Kings,Royal Challengers Bangalore,0,0,4,8.30,NaN,1


## Handling Invalid Run Rate Values

Rows containing NaN or infinite values in the Required Run Rate feature were removed.

These cases occur when the innings has already ended (balls left = 0) and therefore do not represent active match situations suitable for prediction.

In [339]:
final_data[final_data['required_rr'] == np.inf]

,match_id,venue,batting_team,bowling_team,runs_left,balls_left,wickets_left,current_rr,required_rr,result
586,5,M Chinnaswamy Stadium,Delhi Daredevils,Royal Challengers Bangalore,16,0,1,7.10,inf,0
1570,14,Eden Gardens,Sunrisers Hyderabad,Kolkata Knight Riders,18,0,4,7.75,inf,0
1691,15,Feroz Shah Kotla,Kings XI Punjab,Delhi Daredevils,52,0,1,6.85,inf,0
1940,17,M Chinnaswamy Stadium,Royal Challengers Bangalore,Rising Pune Supergiant,28,0,1,6.70,inf,0
2187,19,"Rajiv Gandhi International Stadium, Uppal",Kings XI Punjab,Sunrisers Hyderabad,6,0,0,7.70,inf,0
...,...,...,...,...,...,...,...,...,...,...
69392,616,Dr. Y.S. Rajasekhara Reddy ACA-VDCA Cricket St...,Rising Pune Supergiants,Sunrisers Hyderabad,5,0,2,6.65,inf,0
70546,629,Dr. Y.S. Rajasekhara Reddy ACA-VDCA Cricket St...,Rising Pune Supergiants,Kings XI Punjab,6,0,4,8.35,inf,1
70782,631,Eden Gardens,Sunrisers Hyderabad,Kolkata Knight Riders,23,0,2,7.45,inf,0
71128,634,Feroz Shah Kotla,Kolkata Knight Riders,Sunrisers Hyderabad,23,0,2,7.00,inf,0


In [340]:
second_innings[['runs_left','balls_left','required_rr']].head()

,runs_left,balls_left,required_rr
0,207,119,10.436975
1,207,118,10.525424
2,207,117,10.615385
3,205,116,10.603448
4,201,115,10.486957


In [341]:
# handles the nan and inf values
final_data = final_data.replace([np.inf , -np.inf], np.nan)

# drop the missing values rows
final_data = final_data.dropna()

In [342]:
final_data.shape

(71125, 10)

In [343]:
final_data.isnull().sum()

match_id        0
venue           0
batting_team    0
bowling_team    0
runs_left       0
balls_left      0
wickets_left    0
current_rr      0
required_rr     0
result          0
dtype: int64

# Final Dataset Preparation

After feature engineering, invalid rows containing undefined or infinite Required Run Rate values were removed.

Final Dataset:
- Total Records: 71,125
- Features: 9
- Target Variable: Result
- Missing Values: 0

The dataset is now ready for machine learning model development.

## Data Preprocessing and Feature Encoding

The dataset contains categorical features such as Batting Team, Bowling Team, and Venue. Since machine learning algorithms cannot directly process text data, One Hot Encoding was applied to convert these categorical variables into numerical format.

One Hot Encoding creates separate binary columns for each category and prevents the model from assuming any ordinal relationship between categorical values.

This preprocessing step ensures that the machine learning model can effectively utilize team and venue information during training.

# Model Building

In [424]:
# divide the data in the features and target

x = final_data.drop(columns = ['result'])
y = final_data['result']

In [425]:
print(x.shape)
print(y.shape)

(71125, 9)
(71125,)


In [426]:
from sklearn.model_selection import train_test_split

x_train, x_test, y_train, y_test = train_test_split(x,y,test_size=0.2,random_state=42)

print(x_train.shape)
print(x_test.shape)

## Hot Encoding

In [427]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

In [428]:
# categorial columns
categorical_cols = [
    'batting_team',
    'bowling_team',
    'venue'
]

In [429]:
# hot encoding
numerical_cols = [
    'runs_left',
    'balls_left',
    'wickets_left',
    'current_rr',
    'required_rr'
]

preprocessor = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(drop='first'),
         ['batting_team', 'bowling_team', 'venue']),

        ('num', StandardScaler(),
         numerical_cols)
    ]
)

In [430]:
# pipe lining
pipe = Pipeline([
    ('preprocessor', preprocessor),
    ('model', LogisticRegression(max_iter=1000))
])

## Feature Encoding and Pipeline Construction

Categorical variables were transformed using One Hot Encoding to convert team and venue information into a numerical format.

A machine learning pipeline was created to automate preprocessing and model training. This ensures that the same transformations are consistently applied during both training and prediction.

## Train the model

In [431]:
pipe.fit(x_train, y_train)

,steps,"[('preprocessor', ...), ('model', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('cat', ...), ('num', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [432]:
y_pred = pipe.predict(x_test)

In [433]:
from sklearn.metrics import accuracy_score

accuracy = accuracy_score(y_test, y_pred)

print(accuracy)

0.8241124780316345


# MODEL WITH LOGISTIC REGRESSION 

The Logistic Regression model achieved an accuracy of approximately 82.8% on the test dataset.

This indicates that the engineered features, such as Runs Left, Balls Left, Wickets Left, Current Run Rate, and Required Run Rate, are highly effective in capturing match situations and predicting the outcome of a run chase.

### Confusion Matrix

In [434]:
from sklearn.metrics import confusion_matrix

confusion_matrix(y_test, y_pred)

array([[5453, 1270],
       [1232, 6270]])

### Classification Report

In [435]:
from sklearn.metrics import classification_report

print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.82      0.81      0.81      6723
           1       0.83      0.84      0.83      7502

    accuracy                           0.82     14225
   macro avg       0.82      0.82      0.82     14225
weighted avg       0.82      0.82      0.82     14225



# Model Evaluation

The Logistic Regression model achieved an accuracy of 82.8% on the test dataset.

Confusion Matrix:
- True Negatives: 5500
- False Positives: 1223
- False Negatives: 1223
- True Positives: 6279

Classification Metrics:
- Precision: 0.82 - 0.84
- Recall: 0.82 - 0.84
- F1 Score: 0.82 - 0.84

The model demonstrates balanced performance across both winning and losing match situations, indicating good generalization capability.

# MODEL CHECK WITH RANDOM FOREST 

In [436]:
from sklearn.ensemble import RandomForestClassifier

In [437]:
# pipelining
rf_pipe = Pipeline([
    ('preprocessor', preprocessor),
    ('model', RandomForestClassifier(
        n_estimators=100,
        random_state=42
    ))
])

In [438]:
final_data.columns

Index(['match_id', 'venue', 'batting_team', 'bowling_team', 'runs_left',
       'balls_left', 'wickets_left', 'current_rr', 'required_rr', 'result'],
      dtype='object')

In [439]:
# Fresh trainig kyonki pahle random forest match id ke sath train ho gaya tha 

X = final_data.drop(columns = ['result'])
Y = final_data['result']

In [440]:
# train the random forest model
X_train,X_test,Y_train,Y_test = train_test_split(X,Y,train_size = 0.2,random_state = 42)

In [441]:
rf_pipe.fit(X_train, Y_train)
rf_pred = rf_pipe.predict(X_test)

from sklearn.metrics import accuracy_score

print(accuracy_score(Y_test, rf_pred))

0.9789279437609841


### Random Forest Leakage Investigation

Initial Random Forest results showed extremely high accuracy (>99%). Upon investigation, the `match_id` feature was found to contribute to information leakage. After removing `match_id`, the accuracy decreased but remained unusually high, indicating that ball-level train-test splitting may still allow information from the same match to appear in both training and testing sets.

A match-wise validation strategy will be used to obtain a more realistic estimate of model performance.


In [442]:
match_ids = final_data['match_id'].unique()
print("Total Matches : ",len(match_ids))

Total Matches :  618


In [443]:
# train the model with match wise

train_matches, test_matches = train_test_split(
    match_ids,
    test_size=0.2,
    random_state=42
)

In [444]:
train_df = final_data[
    final_data['match_id'].isin(train_matches)
]

test_df = final_data[
    final_data['match_id'].isin(test_matches)
]

print(train_df.shape)
print(test_df.shape)

(56779, 10)
(14346, 10)


In [445]:
X_train = train_df.drop(columns=['result', 'match_id'])
y_train = train_df['result']

X_test = test_df.drop(columns=['result', 'match_id'])
y_test = test_df['result']

In [446]:
pipe.fit(X_train, y_train)

y_pred = pipe.predict(X_test)

from sklearn.metrics import accuracy_score

logistic_accuracy = accuracy_score(y_test, y_pred)

print(logistic_accuracy)

0.75352014498815


In [447]:
rf_pipe.fit(X_train, y_train)

rf_pred = rf_pipe.predict(X_test)

rf_accuracy = accuracy_score(y_test, rf_pred)

print(rf_accuracy)

0.7993865885961243


Initially, a random train-test split produced higher accuracy scores. However, this approach introduced data leakage because observations from the same match could appear in both training and testing sets.

To obtain a realistic evaluation, a match-wise train-test split was implemented. Using this validation strategy:

- Logistic Regression achieved 74.1% accuracy.
- Random Forest achieved 78.75% accuracy.

Random Forest was selected as the final model due to its superior performance.

# Model Saving

In [448]:
import pickle

pickle.dump(rf_pipe, open('rf_pipe.pkl', 'wb'))

In [449]:
import os

print(os.listdir())

['ipldata.csv', 'Untitled.ipynb', 'rf_pipe.pkl', 'delivery.csv', '.virtual_documents', '.ipynb_checkpoints', 'final_data.pkl']


In [450]:
import os
print('rf_pipe.pkl' in os.listdir())

True


In [451]:
pickle.dump(final_data, open('final_data.pkl', 'wb'))

In [452]:
print(X_train.columns.tolist())

['venue', 'batting_team', 'bowling_team', 'runs_left', 'balls_left', 'wickets_left', 'current_rr', 'required_rr']


In [453]:
import os

print('rf_pipe.pkl' in os.listdir())
print('final_data.pkl' in os.listdir())

True
True
